# Entity disambiguation

Crear nodos unicos de entidades escritas de formas diferentes: (film director/direcor of films, nombres propios escritos con abreviaciones...)

-   Con libreria textdistance se pueden calcular metricas.
-   Con rank-bm25 es como funciona fulltext.queryNode en Neo4j, que funciona bastante bien. Tiene libreria para python

# Imports

In [1]:
import sys
import json
# import spacy
from neo4j import GraphDatabase
import pandas as pd
import numpy as np

In [2]:
from neo4j import GraphDatabase

In [3]:
import bm25s
import textdistance
import regex as re
import spacy

c:\Users\andre\anaconda3\envs\kag_env1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
sys.path.append("../src")

In [5]:
from load_data.funciones_carga_datos import load_filter_dataset_HuggingFace
from conexion_Neo4j.conexion_Neo4j import ConexionNeo4j


In [6]:
# database_Neo = "2wiki.prueba.rebel.3"
# conn_Neo4j = ConexionNeo4j(database_Neo)

In [7]:
driver = GraphDatabase.driver(
    "bolt://localhost:7687",
    auth=("neo4j", "password"),
    database = "2wiki.prueba.rebel.3",
)

In [8]:
def v(df_o_filtro):
    global _v
    _v = df_o_filtro
    return _v

# Rank-BM25

In [6]:
from rank_bm25 import BM25Okapi

## Prueba ejemplo

In [141]:
corpus = [
    "Hello there good man!",
    "It is quite windy in London",
    "How is the weather today?",
    "k. v. mahadevan"
]

tokenized_corpus = [doc.replace(".", " ").replace("  ", " ").split(" ") for doc in corpus]


In [142]:
tokenized_corpus

[['Hello', 'there', 'good', 'man!'],
 ['It', 'is', 'quite', 'windy', 'in', 'London'],
 ['How', 'is', 'the', 'weather', 'today?'],
 ['k', 'v', 'mahadevan']]

In [143]:
bm25 = BM25Okapi(tokenized_corpus)

In [144]:
query = "krishnankoil venkadachalam mahadevan"
tokenized_query = query.replace(".", " ").replace("  ", " ").split(" ")

doc_scores = bm25.get_scores(tokenized_query)
doc_scores

array([0.        , 0.        , 0.        , 0.99682101])

In [152]:
query = "there weather v"
tokenized_query = query.replace(".", " ").replace("  ", " ").split(" ")

doc_scores = bm25.get_scores(tokenized_query)
doc_scores

array([0.89189248, 0.        , 0.80695034, 0.99682101])

In [155]:
maximos = np.argsort(doc_scores)

In [156]:
maximos

array([1, 2, 0, 3])

In [161]:
maximos[-3:]

array([2, 0, 3])

In [162]:
doc_scores[maximos[-3:]]

array([0.80695034, 0.89189248, 0.99682101])

In [167]:
maximos_py = [i.item() for i in maximos[-3:]]
maximos_py

[2, 0, 3]

In [168]:
doc_scores[maximos_py]

array([0.80695034, 0.89189248, 0.99682101])

In [169]:
best_matches = [corpus[i] for i in maximos_py]

In [170]:
best_matches

['How is the weather today?', 'Hello there good man!', 'k. v. mahadevan']

EN FUNCION

In [7]:
def calc_rank_bm25(bm25_index, query, n_max, entidades):
    tokenized_query = query.replace(".", " ").replace("  ", " ").split(" ")
    scores = bm25_index.get_scores(tokenized_query)
    max_indices = np.argsort(scores)[-n_max:]
    max_indices_py = [i.item() for i in max_indices]
    max_scores = scores[max_indices_py]
    max_scores_py = [i.item() for i in max_scores]
    best_matches = [entidades[i] for i in max_indices_py]
    return max_scores_py, best_matches

In [8]:
corpus = [
    "Hello there good man!",
    "It is quite windy in London",
    "How is the weather today?",
    "k. v. mahadevan"
]

tokenized_corpus = [doc.replace(".", " ").replace("  ", " ").split(" ") for doc in corpus]


In [9]:
bm25 = BM25Okapi(tokenized_corpus)

In [10]:
query = "there weather v"

In [11]:
max_scores, best_matches = calc_rank_bm25(bm25, query, 3, corpus)

In [12]:
max_scores

[0.8069503432259082, 0.8918924846181091, 0.9968210122202397]

In [13]:
best_matches

['How is the weather today?', 'Hello there good man!', 'k. v. mahadevan']

BUCLE PARA VARIAS QUERIES . GUARDADO EN df

In [18]:
def calc_rank_bm25(bm25_index, query, n_max, entidades):
    tokenized_query = query.replace(".", " ").replace("  ", " ").split(" ")
    scores = bm25_index.get_scores(tokenized_query)
    max_indices = np.argsort(scores)[-n_max:]
    max_indices_py = [i.item() for i in max_indices]
    max_scores = scores[max_indices_py]
    max_scores_py = [i.item() for i in max_scores]
    best_matches = [entidades[i] for i in max_indices_py]
    return max_scores_py, best_matches

In [ ]:
def rank_bm25_disambiguation(queries, index_bm25, corpus):
    n_max = 3
    filas = []
    for query in queries:
        # print(f"Entidad: {query}")
        max_scores, best_matches = calc_rank_bm25(index_bm25, query, n_max, corpus)
        fila = max_scores + best_matches
        filas.append(fila)
        # print(f"Finaliza: {query}")
    df_bm25 = pd.DataFrame(filas, columns=['score_3_bm25', 'score_2_bm25', 'score_1_bm25', 'similar_3_bm25', 'similar_2_bm25', 'similar_1_bm25'])
    df_bm25['Entidad'] = queries
    cols_reorder = ["Entidad", "similar_1_bm25", "score_1_bm25", "similar_2_bm25", "score_2_bm25", "similar_3_bm25", "score_3_bm25"]
    df_bm25 = df_bm25[cols_reorder]
    return df_bm25

In [20]:
corpus = [
    "Hello there good man!",
    "It is quite windy in London",
    "How is the weather today?",
    "k. v. mahadevan"
]

tokenized_corpus = [doc.replace(".", " ").replace("  ", " ").split(" ") for doc in corpus]


In [21]:
bm25 = BM25Okapi(tokenized_corpus)

In [22]:
queries = ["krishnankoil venkadachalam mahadevan"]

In [26]:
queries = ["krishnankoil venkadachalam mahadevan", "krishnan venkadalam mahavan"]
df_bm25 = rank_bm25_disambiguation(queries, bm25, corpus)

Entidad: krishnankoil venkadachalam mahadevan
Finaliza: krishnankoil venkadachalam mahadevan
Entidad: krishnan venkadalam mahavan
Finaliza: krishnan venkadalam mahavan


In [27]:
df_bm25

,Entidad,similar_1_bm25,score_1_bm25,similar_2_bm25,score_2_bm25,similar_3_bm25,score_3_bm25
0,krishnankoil venkadachalam mahadevan,k. v. mahadevan,0.996821,How is the weather today?,0.0,It is quite windy in London,0.0
1,krishnan venkadalam mahavan,k. v. mahadevan,0.000000,How is the weather today?,0.0,It is quite windy in London,0.0


## CON VARIAS QUERIS - NO FUNCIONA

In [145]:
queries = ["krishnankoil venkadachalam mahadevan", "otra query de prueba"]

In [146]:
queries_tokenized = [qu.replace(".", " ").replace("  ", " ").split(" ") for qu in queries]

In [148]:
doc_scores_queries = bm25.get_scores(queries_tokenized)
doc_scores_queries

TypeError: unhashable type: 'list'

## Con las entidades de Neo4j

In [28]:
entidades = conn_Neo4j.extraer_all_entidades_neo4j()

In [29]:
len(entidades)

4858

In [30]:
def tokenize(text):
    return text.lower().replace(".", "").replace(",", "").replace("'", "").replace("  ", " ").split()


In [31]:
tokenized_corpus = [tokenize(ent) for ent in entidades]

In [32]:
bm25 = BM25Okapi(tokenized_corpus)

In [33]:
query = "krishnankoil venkadachalam mahadevan"
tokenized_query = tokenize(query)

doc_scores = bm25.get_scores(tokenized_query)
doc_scores

array([0., 0., 0., ..., 0., 0., 0.], shape=(4858,))

In [ ]:
df_bm25 = rank_bm25_disambiguation(entidades, bm25, entidades)

In [38]:
df_bm25[df_bm25['Entidad'] != df_bm25['similar_1_bm25']].head(30)

,Entidad,similar_1_bm25,score_1_bm25,similar_2_bm25,score_2_bm25,similar_3_bm25,score_3_bm25
111,mr moto,mr. moto,12.786003,mr moto,12.786003,mysterious mr. moto,10.633322
127,men's tournament,tournament,10.272367,men's tournament,8.192761,"henry lawes luttrell, 2nd earl of carhampton",0.000000
160,spencer's mountain,mountain pass,6.867283,spencer's mountain,6.867283,dark mountain,6.867283
231,catherine & co.,catherine & co,18.869633,catherine & co.,18.869633,catherine,9.201857
307,mohammed bin rashid al maktoum,rashid bin mohammed bin rashid al maktoum,19.030076,mohammed bin rashid al maktoum,18.967419,mohammed bin rashid al,17.413084
395,danger point,point danger,15.167443,danger point,15.167443,moss point,7.338972
421,the falcon's adventure,adventure,9.201857,the falcon's adventure,8.802656,adventure film,7.338972
433,the sword stained with royal blood,sword stained with royal blood,22.745620,the sword stained with royal blood,21.994690,sword of blood and valour,9.019228
473,abm fazle karim chowdhury,a.b.m. fazle karim chowdhury,20.061549,abm fazle karim chowdhury,20.061549,fazle karim chowdhury,16.625748
500,rumbold's cathedral,cathedral,8.610438,birmingham cathedral,6.867283,rumbold's cathedral,6.867283


In [36]:
df_bm25.sort_values("score_2_bm25", ascending=False).head(50)

,Entidad,similar_1_bm25,score_1_bm25,similar_2_bm25,score_2_bm25,similar_3_bm25,score_3_bm25
1108,why did i get married? why did i get married too?,why did i get married? why did i get married too?,43.648970,why did i get married?,40.603622,why did i get married too?,40.387245
1198,rafe kovich and alison barrington kovich,rafe kovich and alison barrington kovich,30.472926,rafe kovich and alison barrington,28.051664,alison skipper,7.828471
37,santa maría de santa cruz de la serós,santa maría de santa cruz de la serós,30.041823,santa cruz de la serós,27.722049,santa maría de oseira,20.944158
601,saeed bin maktoum bin rashid al maktoum,saeed bin maktoum bin rashid al maktoum,29.062498,rashid bin saeed al maktoum,26.945572,maktoum bin rashid al maktoum,26.390121
2181,rashid bin mohammed bin rashid al maktoum,rashid bin mohammed bin rashid al maktoum,28.914491,mohammed bin rashid al maktoum,26.485359,mohammed bin rashid al,26.014368
1400,it goes like it goes,it goes like it goes,31.861867,it goes like this,25.043057,so it goes,23.155997
285,majid bin mohammed bin rashid al maktoum,majid bin mohammed bin rashid al maktoum,25.735508,rashid bin mohammed bin rashid al maktoum,23.707535,mohammed bin rashid al maktoum,22.525025
475,hind bint maktoum bin juma al maktoum,hind bint maktoum bin juma al maktoum,26.674966,maktoum bin juma al maktoum,23.573317,hind bint maktoum,22.443076
731,the taking of pelham one two three the convers...,the taking of pelham one two three the convers...,25.096983,the taking of pelham one two three,23.554447,three days of the condor,10.139239
3725,david morris and jacqui morris,david morris and jacqui morris,26.773751,jacqui morris,23.305406,david morris,21.652188


# BM25s

Lo mismo que rank-bm25 pero mucho mas rapido y con funcionalidades extra. Trae tokenizador, se puede guardar el index...

In [6]:
import bm25s

In [7]:
entidades = conn_Neo4j.extraer_all_entidades_neo4j()
len(entidades)

4858

In [8]:
entidades_tokens = bm25s.tokenize(entidades, stopwords="en")

In [9]:
retriever = bm25s.BM25()
retriever.index(entidades_tokens)

In [10]:
query = "krishnankoil venkadachalam mahadevan"
query_tokens = bm25s.tokenize(query)


In [11]:
# Get top-k results as a tuple of (doc ids, scores). Both are arrays of shape (n_queries, k).
# To return docs instead of IDs, set the `corpus=corpus` parameter.
results, scores = retriever.retrieve(query_tokens, k=4)

In [12]:
scores

array([[7.797125 , 6.3085027, 4.0111074, 0.       ]], dtype=float32)

In [13]:
results

array([[ 622, 2106, 3579,   31]])

In [14]:
print(entidades[622])
print(entidades[2106])
print(entidades[3579])
print(entidades[31])

krishnankoil venkadachalam mahadevan
krishnankoil venkadachalam
k. v. mahadevan
henry lawes luttrell, 2nd earl of carhampton


In [15]:
for i in range(results.shape[1]):
    doc, score = results[0, i], scores[0, i]
    print(f"Rank {i+1} (score: {score:.2f}): {doc}")

Rank 1 (score: 7.80): 622
Rank 2 (score: 6.31): 2106
Rank 3 (score: 4.01): 3579
Rank 4 (score: 0.00): 31


## Con todas las entidades

In [16]:
entidades_token = bm25s.tokenize(entidades, stopwords="en")

In [18]:
# results, scores = retriever.retrieve(entidades_token, k=4)
results, scores = retriever.retrieve(entidades_token, k=3)

In [19]:
results

array([[   0, 2169, 3346],
       [   1, 2071,  551],
       [   2,  581,  341],
       ...,
       [4855, 1872,   31],
       [4856, 4849, 4848],
       [4857, 4849, 4848]], shape=(4858, 3))

In [20]:
scores

array([[7.647942 , 4.0111074, 3.3039467],
       [5.9800696, 3.1542513, 2.8258183],
       [6.9683337, 6.061139 , 5.115469 ],
       ...,
       [4.0111074, 3.1542513, 0.       ],
       [4.2816963, 0.       , 0.       ],
       [4.2816963, 0.       , 0.       ]], shape=(4858, 3), dtype=float32)

### Guardado resultados en un df de pandas

In [21]:
# df_resultados = pd.DataFrame(data=results, columns=["similar_1", "similar_2", "similar_3", "similar_4"])
df_resultados = pd.DataFrame(data=results, columns=["similar_1", "similar_2", "similar_3"])

In [22]:
# df_scores = pd.DataFrame(scores, columns=["score_1", "score_2", "score_3", "score_4"])
df_scores = pd.DataFrame(scores, columns=["score_1", "score_2", "score_3"])

In [23]:
df_resultados = df_resultados.map(lambda x: entidades[x])

In [24]:
df_final = pd.concat([df_resultados, df_scores], axis=1)

In [25]:
df_final['Entidad'] = entidades

In [26]:
# cols_reorder = ["Entidad", "similar_1", "score_1", "similar_2", "score_2", "similar_3", "score_3", "similar_4", "score_4"]
cols_reorder = ["Entidad", "similar_1", "score_1", "similar_2", "score_2", "similar_3", "score_3"]

In [27]:
df_final = df_final[cols_reorder]

### Revision resultados

In [28]:
print(entidades[1820])
print(entidades[1357])

tang-e khoshk
tang -e khoshk


In [29]:
df_final[df_final['Entidad'] == 'movies']

,Entidad,similar_1,score_1,similar_2,score_2,similar_3,score_3
4108,movies,movies,3.303947,at the movies,3.303947,merton of the movies,2.598155


In [30]:
print(entidades[449])
print(entidades[4108])

at the movies
movies


In [31]:
print(entidades[1571])
print(entidades[2163])

do you remember dolly bell?
do you remember dolly bell


In [32]:
results[1571]

array([2163, 1571, 1385])

In [33]:
df_final[df_final['score_1'] != df_final['score_2']].sort_values('score_1', ascending = False)

,Entidad,similar_1,score_1,similar_2,score_2,similar_3,score_3
1108,why did i get married? why did i get married too?,why did i get married? why did i get married too?,17.135550,why did i get married,16.190729,why did i get married?,16.190729
1219,"you're my heart, you're my soul'98 china in he...","you're my heart, you're my soul'98 china in he...",14.260234,"you're my love, you're my life",11.263678,you're my pet,10.647292
1499,"you're my love, you're my life","you're my love, you're my life",13.546166,you're my pet,10.647292,you're my boss,10.647292
2230,"ich für dich, du für mich","ich für dich, du für mich",12.588615,sms für dich,7.797125,les amants du pont-neuf,1.922313
1198,rafe kovich and alison barrington kovich,rafe kovich and alison barrington kovich,11.813978,rafe kovich and alison barrington,10.951963,alison skipper,3.014093
...,...,...,...,...,...,...,...
3974,river,river,2.802111,colorado river,2.203521,kouga river,2.203521
2839,province,province,2.760544,mazandaran province,2.170834,markazi province,2.170834
2758,march,march,2.686077,march 1963,2.112275,"march 3, 1981",2.112275
3557,airport,airport,2.576631,erzincan airport,2.026209,international airport,2.026209


In [34]:
# df_final[df_final['score_1'] > 6]
df_final[df_final['score_2'].between(5,6)].sort_values('score_2', ascending = False).head(60)

,Entidad,similar_1,score_1,similar_2,score_2,similar_3,score_3
590,marie louise coidavid,marie louise coidavid,7.324757,louise coidavid,5.980070,louise rousseau,2.825818
3866,"cedar park, texas","cedar park, texas",7.411015,cedar park,5.980070,texas,3.832876
2981,"moss point, mississippi","moss point, mississippi",7.411015,moss point,5.980070,mississippi,3.832876
1858,curly ray cline,curly ray cline,7.701834,ray cline,5.980070,ray mccarey,2.825818
1820,tang-e khoshk,tang-e khoshk,5.980070,tang -e khoshk,5.980070,tang dynasty,2.825818
1357,tang -e khoshk,tang-e khoshk,5.980070,tang -e khoshk,5.980070,tang dynasty,2.825818
1432,homonymous album of the year 2011,homonymous album of the year 2011,8.480229,homonymous album,5.980070,album,3.593455
1277,belcher channel formation,belcher channel formation,7.411015,belcher channel,5.980070,formation,3.832876
1968,silent film era,silent film era,6.232367,silent era,5.980070,silent film,4.409479
1983,there is so much world to see,there is so much world to see,8.059259,so much,5.980070,medeweger see,3.154251


In [35]:
df_final[df_final['score_2'] > 6].sort_values('Entidad', ascending = False)[0:5]

,Entidad,similar_1,score_1,similar_2,score_2,similar_3,score_3
154,åke leonard lindman,åke leonard lindman,7.479892,åke lindman,6.168344,åke leonard järvinen,4.880850
1163,zayed bin sultan al nahyan,zayed bin sultan al nahyan,8.130172,sheikh zayed bin sultan al nahyan,7.193644,mohammed bin rashid al,2.888497
1499,"you're my love, you're my life","you're my love, you're my life",13.546166,you're my pet,10.647292,you're my boss,10.647292
1219,"you're my heart, you're my soul'98 china in he...","you're my heart, you're my soul'98 china in he...",14.260234,"you're my love, you're my life",11.263678,you're my pet,10.647292
178,women's 4 × 400 metres relay event,women's 4 × 400 metres relay event,9.070564,women's 4 × 400 metres relay,8.218159,400 metres event,7.393633


In [36]:
df_final[df_final['score_2'] > 6].sort_values('score_2', ascending = False)[0:50]

,Entidad,similar_1,score_1,similar_2,score_2,similar_3,score_3
1108,why did i get married? why did i get married too?,why did i get married? why did i get married too?,17.135550,why did i get married,16.190729,why did i get married?,16.190729
1219,"you're my heart, you're my soul'98 china in he...","you're my heart, you're my soul'98 china in he...",14.260234,"you're my love, you're my life",11.263678,you're my pet,10.647292
1198,rafe kovich and alison barrington kovich,rafe kovich and alison barrington kovich,11.813978,rafe kovich and alison barrington,10.951963,alison skipper,3.014093
1499,"you're my love, you're my life","you're my love, you're my life",13.546166,you're my pet,10.647292,you're my boss,10.647292
37,santa maría de santa cruz de la serós,santa maría de santa cruz de la serós,11.260831,santa cruz de la serós,10.415339,santa maría de oseira,7.920996
601,saeed bin maktoum bin rashid al maktoum,saeed bin maktoum bin rashid al maktoum,10.854058,rashid bin saeed al maktoum,10.014881,saeed bin maktoum,9.939018
2181,rashid bin mohammed bin rashid al maktoum,rashid bin mohammed bin rashid al maktoum,10.800768,mohammed bin rashid al maktoum,9.841379,mohammed bin rashid al,9.688490
3159,"new york, new york","new york, new york",10.326508,new york,9.519973,new york city,7.844273
3725,david morris and jacqui morris,david morris and jacqui morris,10.286463,jacqui morris,8.973067,david morris,8.337386
439,why did i get married too?,why did i get married? why did i get married too?,9.199800,why did i get married too?,8.963757,why did i get married,8.095365


In [37]:
df_final[df_final['Entidad'] == "why did i get married?"].sort_values('score_2', ascending = False)[0:50]

,Entidad,similar_1,score_1,similar_2,score_2,similar_3,score_3
454,why did i get married?,why did i get married,8.095365,why did i get married?,8.095365,why did i get married? why did i get married too?,7.935747


In [38]:
df_final[df_final['Entidad'] == "you're my pet"].sort_values('score_2', ascending = False)[0:50]

,Entidad,similar_1,score_1,similar_2,score_2,similar_3,score_3
1347,you're my pet,you're my pet,7.682767,"you're my love, you're my life",5.631839,you're my boss,5.323646


CALCULO BM25s INVERSO - FUNCIONA OK

In [ ]:

def calcular_score_inverso(fila, retriever, entidades):

    similares = [fila['similar_1'], fila['similar_2'], fila['similar_3']]
    similares_tokenized = bm25s.tokenize(similares)
    
    # 2. Buscamos en el retriever. Pedimos los top 10 o 20 resultados 
    # para asegurarnos de encontrar la entidad original (A) entre ellos
    resultados, scores = retriever.retrieve(similares_tokenized, k=20)
    
    idx_entidad = entidades.index(fila['Entidad'])

    scores_inv = []
    for i, res in enumerate(resultados):
        if idx_entidad in res:
            idx_entidad_resultado = res.tolist().index(idx_entidad)
            score = scores[i][idx_entidad_resultado]
            scores_inv.append(score)
        else:
            scores_inv.append(0)

    return scores_inv[0], scores_inv[1], scores_inv[2]

In [40]:
df_final[['score_1_inv', 'score_2_inv', 'score_3_inv']] = df_final.apply(calcular_score_inverso, args = (retriever, entidades), axis=1, result_type='expand')

In [41]:
df_final[df_final['score_2'] > 6].sort_values('score_2', ascending = False)[0:50]

,Entidad,similar_1,score_1,similar_2,score_2,similar_3,score_3,score_1_inv,score_2_inv,score_3_inv
1108,why did i get married? why did i get married too?,why did i get married? why did i get married too?,17.135550,why did i get married,16.190729,why did i get married?,16.190729,17.135550,7.935747,7.935747
1219,"you're my heart, you're my soul'98 china in he...","you're my heart, you're my soul'98 china in he...",14.260234,"you're my love, you're my life",11.263678,you're my pet,10.647292,14.260234,8.554621,4.277310
1198,rafe kovich and alison barrington kovich,rafe kovich and alison barrington kovich,11.813978,rafe kovich and alison barrington,10.951963,alison skipper,3.014093,11.813978,8.747749,1.836896
1499,"you're my love, you're my life","you're my love, you're my life",13.546166,you're my pet,10.647292,you're my boss,10.647292,13.546166,5.631839,5.631839
37,santa maría de santa cruz de la serós,santa maría de santa cruz de la serós,11.260831,santa cruz de la serós,10.415339,santa maría de oseira,7.920996,11.260831,6.898088,4.362741
601,saeed bin maktoum bin rashid al maktoum,saeed bin maktoum bin rashid al maktoum,10.854058,rashid bin saeed al maktoum,10.014881,saeed bin maktoum,9.939018,10.854058,7.212465,5.099012
2181,rashid bin mohammed bin rashid al maktoum,rashid bin mohammed bin rashid al maktoum,10.800768,mohammed bin rashid al maktoum,9.841379,mohammed bin rashid al,9.688490,10.800768,7.053528,5.931471
3159,"new york, new york","new york, new york",10.326508,new york,9.519973,new york city,7.844273,10.326508,5.163254,5.163254
3725,david morris and jacqui morris,david morris and jacqui morris,10.286463,jacqui morris,8.973067,david morris,8.337386,10.286463,5.365927,4.920536
439,why did i get married too?,why did i get married? why did i get married too?,9.199800,why did i get married too?,8.963757,why did i get married,8.095365,16.005199,8.963757,7.041444


In [43]:
df_final[df_final['Entidad'] == 'rosie and the goldbug']

,Entidad,similar_1,score_1,similar_2,score_2,similar_3,score_3,score_1_inv,score_2_inv,score_3_inv
1872,rosie and the goldbug,rosie and the goldbug,6.168344,goldbug,4.011107,rosie,3.832876,6.168344,3.154251,3.014093


# Distancias entre textos

In [44]:
import textdistance

In [51]:
def calcular_jaro_winkler(fila):
    dist_jaro_1 = textdistance.jaro_winkler(fila['Entidad'], fila['similar_1'])
    dist_jaro_2 = textdistance.jaro_winkler(fila['Entidad'], fila['similar_2'])
    dist_jaro_3 = textdistance.jaro_winkler(fila['Entidad'], fila['similar_3'])
    return dist_jaro_1, dist_jaro_2, dist_jaro_3

In [60]:
def calcular_levenshtein(fila):
    dist_leven_1 = textdistance.levenshtein.normalized_similarity(fila['Entidad'], fila['similar_1'])
    dist_leven_2 = textdistance.levenshtein.normalized_similarity(fila['Entidad'], fila['similar_2'])
    dist_leven_3 = textdistance.levenshtein.normalized_similarity(fila['Entidad'], fila['similar_3'])
    return dist_leven_1, dist_leven_2, dist_leven_3

In [52]:
df_final[['dist_jaro_1', 'dist_jaro_2', 'dist_jaro_3']] = df_final.apply(calcular_jaro_winkler, axis = 1, result_type='expand')

In [61]:
df_final[['dist_leven_1', 'dist_leven_2', 'dist_leven_3']] = df_final.apply(calcular_levenshtein, axis = 1, result_type='expand')

In [56]:
df_final['Entidad'][1219]

"you're my heart, you're my soul'98 china in her eyes"

In [63]:
df_final.columns

Index(['Entidad', 'similar_1', 'score_1', 'similar_2', 'score_2', 'similar_3',
       'score_3', 'score_1_inv', 'score_2_inv', 'score_3_inv', 'dist_jaro_1',
       'dist_jaro_2', 'dist_jaro_3', 'dist_leven_1', 'dist_leven_2',
       'dist_leven_3'],
      dtype='object')

In [64]:
cols_ = ['Entidad', 'similar_1', 'score_1', 'score_1_inv', 'dist_jaro_1', 'dist_leven_1', 'similar_2', 'score_2', 'score_2_inv', 'dist_jaro_2', 'dist_leven_2']

In [65]:
df_final[df_final['score_2'] > 6][cols_].sort_values('score_2', ascending = False)[0:50]

,Entidad,similar_1,score_1,score_1_inv,dist_jaro_1,dist_leven_1,similar_2,score_2,score_2_inv,dist_jaro_2,dist_leven_2
1108,why did i get married? why did i get married too?,why did i get married? why did i get married too?,17.135550,17.135550,1.000000,1.000000,why did i get married,16.190729,7.935747,0.885714,0.428571
1219,"you're my heart, you're my soul'98 china in he...","you're my heart, you're my soul'98 china in he...",14.260234,14.260234,1.000000,1.000000,"you're my love, you're my life",11.263678,8.554621,0.831994,0.480769
1198,rafe kovich and alison barrington kovich,rafe kovich and alison barrington kovich,11.813978,11.813978,1.000000,1.000000,rafe kovich and alison barrington,10.951963,8.747749,0.965000,0.825000
1499,"you're my love, you're my life","you're my love, you're my life",13.546166,13.546166,1.000000,1.000000,you're my pet,10.647292,5.631839,0.842564,0.366667
37,santa maría de santa cruz de la serós,santa maría de santa cruz de la serós,11.260831,11.260831,1.000000,1.000000,santa cruz de la serós,10.415339,6.898088,0.864373,0.594595
601,saeed bin maktoum bin rashid al maktoum,saeed bin maktoum bin rashid al maktoum,10.854058,10.854058,1.000000,1.000000,rashid bin saeed al maktoum,10.014881,7.212465,0.695651,0.512821
2181,rashid bin mohammed bin rashid al maktoum,rashid bin mohammed bin rashid al maktoum,10.800768,10.800768,1.000000,1.000000,mohammed bin rashid al maktoum,9.841379,7.053528,0.788347,0.731707
3159,"new york, new york","new york, new york",10.326508,10.326508,1.000000,1.000000,new york,9.519973,5.163254,0.888889,0.444444
3725,david morris and jacqui morris,david morris and jacqui morris,10.286463,10.286463,1.000000,1.000000,jacqui morris,8.973067,5.365927,0.664103,0.433333
439,why did i get married too?,why did i get married? why did i get married too?,9.199800,16.005199,0.890738,0.530612,why did i get married too?,8.963757,8.963757,1.000000,1.000000


In [68]:
df_final[(df_final['dist_jaro_2'].between(0.9, 0.999) ) & (df_final['dist_leven_2'].between(0.9, 0.999) )][cols_].sort_values('score_2', ascending = False)[0:50]

,Entidad,similar_1,score_1,score_1_inv,dist_jaro_1,dist_leven_1,similar_2,score_2,score_2_inv,dist_jaro_2,dist_leven_2
2163,do you remember dolly bell,do you remember dolly bell,8.733723,8.733723,1.000000,1.000000,do you remember dolly bell?,8.733723,8.733723,0.992593,0.962963
1899,another latin love song/miss world,another latin love song/miss world,8.637592,8.637592,1.000000,1.000000,another latin love song/ miss world,8.637592,8.637592,0.982521,0.971429
1854,why did i get married,why did i get married,8.095365,8.095365,1.000000,1.000000,why did i get married?,8.095365,8.095365,0.990909,0.954545
4586,"grande- anse, new brunswick","grande- anse, new brunswick",8.002451,8.002451,1.000000,1.000000,"grande-anse, new brunswick",8.002451,8.002451,0.954131,0.962963
3669,united arab emirates(uae,united arab emirates(uae,7.794371,7.794371,1.000000,1.000000,united arab emirates(uae),7.794371,7.794371,0.992000,0.960000
1771,sarre anglo-saxon cemetery,sarre anglo-saxon cemetery,7.790196,7.790196,1.000000,1.000000,sarre anglo- saxon cemetery,7.790196,7.790196,0.969516,0.962963
2340,william marshall(died 1540),william marshall(died 1540),7.742723,7.742723,1.000000,1.000000,william marshall(died 1540?),7.742723,7.742723,0.992857,0.964286
4166,warmian-masurian voivodeship,warmian-masurian voivodeship,7.681638,7.681638,1.000000,1.000000,warmian- masurian voivodeship,7.681638,7.681638,0.964532,0.965517
2505,mr. moto takes a chance,mr. moto takes a chance,7.549897,7.549897,1.000000,1.000000,mr moto takes a chance,7.549897,7.549897,0.988406,0.956522
4815,eleni zaude gabre-,eleni zaude gabre-,7.450663,7.450663,1.000000,1.000000,eleni zaude gabre,7.450663,7.450663,0.988889,0.944444


# Excluir fechas, numeros romanos y numeros al lado de texto de la fusion

In [69]:
import regex as re

In [71]:
def excluir_de_fusion(fila):
    excluir = False
    regex_romanos = r'\b(v[ii]{0,3}|i[vx]|x[lcvi]{0,4}|i{1,3})\b'
    romanos_a = set(re.findall(regex_romanos, fila['Entidad']))
    romanos_b = set(re.findall(regex_romanos, fila['similar_2']))
    if romanos_a and romanos_b and romanos_a != romanos_b:
        excluir = True
    numeros_a = set(re.findall(r'\d+', fila['Entidad']))
    numeros_b = set(re.findall(r'\d+', fila['similar_2']))

    if numeros_a and numeros_b and numeros_a != numeros_b:
        excluir = True
    return excluir

In [74]:
df_final['excluir_2'] = df_final.apply(excluir_de_fusion, axis = 1)

In [84]:
cols_ = ['Entidad', 'similar_1', 'score_1', 'score_1_inv', 'dist_jaro_1', 'dist_leven_1', 'similar_2', 'score_2', 'score_2_inv', 'dist_jaro_2', 'dist_leven_2', 'excluir_2']

In [88]:
df_final[
    (df_final['dist_jaro_2'].between(0.85, 0.999)) &
    (df_final['dist_leven_2'].between(0.85, 0.999)) &
    (df_final['excluir_2'] == False)
    ][cols_].sort_values('score_2', ascending = False)

,Entidad,similar_1,score_1,score_1_inv,dist_jaro_1,dist_leven_1,similar_2,score_2,score_2_inv,dist_jaro_2,dist_leven_2,excluir_2
285,majid bin mohammed bin rashid al maktoum,majid bin mohammed bin rashid al maktoum,9.564405,9.564405,1.0,1.0,rashid bin mohammed bin rashid al maktoum,8.827232,9.224735,0.897540,0.926829,False
2163,do you remember dolly bell,do you remember dolly bell,8.733723,8.733723,1.0,1.0,do you remember dolly bell?,8.733723,8.733723,0.992593,0.962963,False
1899,another latin love song/miss world,another latin love song/miss world,8.637592,8.637592,1.0,1.0,another latin love song/ miss world,8.637592,8.637592,0.982521,0.971429,False
1854,why did i get married,why did i get married,8.095365,8.095365,1.0,1.0,why did i get married?,8.095365,8.095365,0.990909,0.954545,False
4586,"grande- anse, new brunswick","grande- anse, new brunswick",8.002451,8.002451,1.0,1.0,"grande-anse, new brunswick",8.002451,8.002451,0.954131,0.962963,False
...,...,...,...,...,...,...,...,...,...,...,...,...
1088,luciano orodisio,luciano orodisio,6.381130,6.381130,1.0,1.0,luciano odorisio,3.014093,3.014093,0.987500,0.875000,False
4030,nandi awards,nandi awards,6.028187,6.028187,1.0,1.0,nandi award,3.014093,3.014093,0.983333,0.916667,False
3197,mass cycles,mass cycles,6.381130,6.381130,1.0,1.0,mass cycle,3.014093,3.014093,0.981818,0.909091,False
680,trade register,trade register,6.063659,6.063659,1.0,1.0,trade registers,2.909408,2.909408,0.986667,0.933333,False


In [120]:
pd.set_option('display.max_rows', 50) # Quita el límite de filas


In [92]:
display(df_final[
    (df_final['excluir_2'] == True)][["Entidad", "similar_2"]])

,Entidad,similar_2
25,theodred ii,theodred i
31,"henry lawes luttrell, 2nd earl of carhampton","simon luttrell, 1st earl of carhampton"
237,rhoemetalces i,rhoemetalces ii
409,flight nh692,boeing 787(flight nh692)
482,1999 cannes film festival,1985 cannes film festival
596,"simon luttrell, 1st earl of carhampton","henry lawes luttrell, 2nd earl of carhampton"
653,9th berlin international film festival,12th berlin international film festival
755,11th berlin international film festival,12th berlin international film festival
768,cotys iii,cotys i
817,cotys vi,cotys i


# Excluir cambios con terminos poco frecuentes

NO FUNCIONA .+- SON CASI TODOS POCO FRECUENTES

In [154]:
from sklearn.feature_extraction.text import TfidfVectorizer


coleccion = [
    "procesador memoria_ram tarjeta_grafica",
    "procesador memoria_ram disco_duro fuente_poder",
    "procesador memoria_ram",
    "procesador placa_base refrigeracion_liquida",
]

# 2. Inicializamos el vectorizador
# Usamos token_pattern para que acepte guiones bajos si tus entidades los tienen
vectorizer = TfidfVectorizer(stop_words="english")


In [155]:

# 3. Ajustamos el modelo a los datos
tfidf_matrix = vectorizer.fit_transform(entidades)


In [156]:
entidades_vectorizer = vectorizer.get_feature_names_out()
valores_idf = vectorizer.idf_

In [157]:
entidades_vectorizer

array(['10', '100', '11', ..., 'šlomović', 'žižek', '濰水之戰'],
      shape=(4786,), dtype=object)

In [158]:
valores_idf

array([6.39754548, 8.38997565, 6.65537459, ..., 8.79544075, 8.79544075,
       8.79544075], shape=(4786,))

In [160]:
idf_df = pd.DataFrame()

In [ ]:
data=[entidades_vectorizer, valores_idf], columns=['entidades', 'idf']

In [161]:
idf_df['entidades'] = entidades_vectorizer
idf_df['idf'] = valores_idf

In [ ]:
idf_df[idf_df['entidades'] == '']

In [162]:
idf_df

,entidades,idf
0,10,6.397545
1,100,8.389976
2,11,6.655375
3,11th,8.795441
4,12,7.003681
...,...,...
4781,élise,8.389976
4782,łódź,8.795441
4783,šlomović,8.795441
4784,žižek,8.795441


# EXCLUIR CUANDO SOLO CAMBIE 1 PALABRA Y SEA UN NOMNRE PROPIO, CIUDAD...

In [ ]:
# Cargamos el modelo ligero en inglés
nlp = spacy.load("en_core_web_sm")



In [ ]:
def es_falso_positivo_por_nombre(fila):
    
    if len(fila['Entidad'].strip()) == len(fila['similar_2'].strip()):
        doc1 = nlp(fila['Entidad'])
        doc2 = nlp(fila['similar_2'])
        
        ents_dict1 = {ent.text: ent.label_ for ent in doc1.ents}
        ents_dict2 = {ent.text: ent.label_ for ent in doc2.ents}

        palabras_str1 = {ent.text for ent in doc1.ents}
        palabras_str2 = {ent.text for ent in doc2.ents}

        diferencias = palabras_str1.symmetric_difference(palabras_str2)
        labels_excluir = ["PERSON", "GPE"]

        for palabra_dif in diferencias:
            label_1 = ents_dict1.get(palabra_dif)
            label_2 = ents_dict2.get(palabra_dif)
            if label_1 in labels_excluir or label_2 in labels_excluir:
                return True

    return False

In [270]:
df_final['excluir_por_nombre_propio'] = df_final.apply(es_falso_positivo_por_nombre, axis=1)

In [187]:
df_final_excluidos_nombres = df_final[df_final['excluir_por_nombre_propio'] == True]

In [271]:
df_final_excluidos_nombres = df_final[
    (df_final['dist_jaro_2'].between(0.7, 0.999)) &
    (df_final['dist_leven_2'].between(0.7, 0.999)) &
    (df_final['score_2'] > 5) &
    (df_final['excluir_por_nombre_propio'] == True)
    ][cols_].sort_values('score_2', ascending = False)

In [272]:
df_final_excluidos_nombres

,Entidad,similar_1,score_1,score_1_inv,dist_jaro_1,dist_leven_1,similar_2,score_2,score_2_inv,dist_jaro_2,dist_leven_2,excluir_2
1744,roman catholic archdiocese of boston,roman catholic archdiocese of boston,7.995131,7.995131,1.0,1.0,roman catholic archdiocese of newark,5.883300,5.883300,0.944444,0.833333,False
2618,roman catholic archdiocese of newark,roman catholic archdiocese of newark,8.242422,8.242422,1.0,1.0,roman catholic archdiocese of boston,5.883300,5.883300,0.944444,0.833333,False
1325,12th moscow international film festival,12th moscow international film festival,7.178578,7.178578,1.0,1.0,12th berlin international film festival,5.498832,5.498832,0.859674,0.846154,False
2206,12th berlin international film festival,12th berlin international film festival,7.178578,7.178578,1.0,1.0,12th moscow international film festival,5.498832,5.498832,0.859674,0.846154,False
755,11th berlin international film festival,11th berlin international film festival,7.308257,7.308257,1.0,1.0,12th berlin international film festival,5.256265,5.256265,0.984615,0.974359,True
2575,8th moscow international film festival,8th moscow international film festival,6.978419,6.978419,1.0,1.0,3rd moscow international film festival,5.256265,5.256265,0.861654,0.921053,True
2306,beshr ibn hasan,beshr ibn hasan,6.678320,6.678320,1.0,1.0,hasan ibn hasan,5.109236,6.185756,0.700000,0.733333,False


# Calculo completo

In [197]:
def tokenize(text):
    return text.lower().replace(".", "").replace(",", "").replace("'", "").replace("  ", " ").split()

In [198]:
def calcular_rank_B25s(entidades):
   entidades_token = bm25s.tokenize(entidades, stopwords="en")
   retriever = bm25s.BM25()
   retriever.index(entidades_token)
   results, scores = retriever.retrieve(entidades_token, k=3)
   df_resultados = pd.DataFrame(data=results, columns=["similar_1", "similar_2", "similar_3"])
   df_scores = pd.DataFrame(scores, columns=["score_1", "score_2", "score_3"])
   df_resultados = df_resultados.map(lambda x: entidades[x])
   df_final = pd.concat([df_resultados, df_scores], axis=1)
   df_final['Entidad'] = entidades
   cols_reorder = ["Entidad", "similar_1", "score_1", "similar_2", "score_2", "similar_3", "score_3"]
   df_final = df_final[cols_reorder]
   return df_final

In [199]:
# def calcular_score_inverso(fila, retriever, entidades):

#     similares = [fila['similar_1'], fila['similar_2'], fila['similar_3']]
#     similares_tokenized = bm25s.tokenize(similares)

#     resultados, scores = retriever.retrieve(similares_tokenized, k=20)
    
#     idx_entidad = entidades.index(fila['Entidad'])

#     scores_inv = []
#     for i, res in enumerate(resultados):
#         if idx_entidad in res:
#             idx_entidad_resultado = res.tolist().index(idx_entidad)
#             score = scores[i][idx_entidad_resultado]
#             scores_inv.append(score)
#         else:
#             scores_inv.append(0)

#     return scores_inv[0], scores_inv[1], scores_inv[2]

In [200]:
# def get_spacy_docs(fila):
    
#     nlp = spacy.load("en_core_web_sm")
#     doc_ent = nlp(fila['Entidad'])
#     doc_sim_1 = nlp(fila['similar_1'])
#     doc_sim_2 = nlp(fila['similar_2'])
#     doc_sim_3 = nlp(fila['similar_3'])
#     return doc_ent, doc_sim_1, doc_sim_2, doc_sim_3
""""
TARDA MUCHO Y PETA LA MEMORIA - IMPOSIBLE GUARDAR OBJETOS DE SPACY EN COLUMNAS DE PANDAS
"""

'"\nTARDA MUCHO Y PETA LA MEMORIA - IMPOSIBLE GUARDAR OBJETOS DE SPACY EN COLUMNAS DE PANDAS\n'

In [201]:
def calcular_jaro_winkler(fila):
    # dist_jaro_1 = textdistance.jaro_winkler(fila['Entidad'], fila['similar_1'])
    # dist_jaro_2 = textdistance.jaro_winkler(fila['Entidad'], fila['similar_2'])
    # dist_jaro_3 = textdistance.jaro_winkler(fila['Entidad'], fila['similar_3'])
    
    doc_ent = nlp(fila['Entidad'])
    doc_sim_1 = nlp(fila['similar_1'])
    doc_sim_2 = nlp(fila['similar_2'])
    doc_sim_3 = nlp(fila['similar_3'])
    
    ent_sw_removed = " ".join([token.text for token in doc_ent if not token.is_stop and not token.is_punct])
    sim_1_sw_removed = " ".join([token.text for token in doc_sim_1 if not token.is_stop and not token.is_punct])
    sim_2_sw_removed = " ".join([token.text for token in doc_sim_2 if not token.is_stop and not token.is_punct])
    sim_3_sw_removed = " ".join([token.text for token in doc_sim_3 if not token.is_stop and not token.is_punct])
    
    dist_jaro_1 = textdistance.jaro_winkler(ent_sw_removed, sim_1_sw_removed)
    dist_jaro_2 = textdistance.jaro_winkler(ent_sw_removed, sim_2_sw_removed)
    dist_jaro_3 = textdistance.jaro_winkler(ent_sw_removed, sim_3_sw_removed)
    return dist_jaro_1, dist_jaro_2, dist_jaro_3

In [202]:
def calcular_levenshtein(fila):
    # dist_leven_1 = textdistance.levenshtein.normalized_similarity(fila['Entidad'], fila['similar_1'])
    # dist_leven_2 = textdistance.levenshtein.normalized_similarity(fila['Entidad'], fila['similar_2'])
    # dist_leven_3 = textdistance.levenshtein.normalized_similarity(fila['Entidad'], fila['similar_3'])
    
    doc_ent = nlp(fila['Entidad'])
    doc_sim_1 = nlp(fila['similar_1'])
    doc_sim_2 = nlp(fila['similar_2'])
    doc_sim_3 = nlp(fila['similar_3'])
    
    ent_sw_removed = " ".join([token.text for token in doc_ent if not token.is_stop and not token.is_punct])
    sim_1_sw_removed = " ".join([token.text for token in doc_sim_1 if not token.is_stop and not token.is_punct])
    sim_2_sw_removed = " ".join([token.text for token in doc_sim_2 if not token.is_stop and not token.is_punct])
    sim_3_sw_removed = " ".join([token.text for token in doc_sim_3 if not token.is_stop and not token.is_punct])
    
    dist_leven_1 = textdistance.levenshtein.normalized_similarity(ent_sw_removed, sim_1_sw_removed)
    dist_leven_2 = textdistance.levenshtein.normalized_similarity(ent_sw_removed, sim_2_sw_removed)
    dist_leven_3 = textdistance.levenshtein.normalized_similarity(ent_sw_removed, sim_3_sw_removed)
    
    return dist_leven_1, dist_leven_2, dist_leven_3

In [203]:
def calcular_jaccard(fila):
    
    # Filtrar stop words y puntuacion
    doc_ent = nlp(fila['Entidad'])
    doc_sim_1 = nlp(fila['similar_1'])
    doc_sim_2 = nlp(fila['similar_2'])
    doc_sim_3 = nlp(fila['similar_3'])
    
    ent_sw_removed = [token.text for token in doc_ent if not token.is_stop and not token.is_punct]
    sim_1_sw_removed = [token.text for token in doc_sim_1 if not token.is_stop and not token.is_punct]
    sim_2_sw_removed = [token.text for token in doc_sim_2 if not token.is_stop and not token.is_punct]
    sim_3_sw_removed = [token.text for token in doc_sim_3 if not token.is_stop and not token.is_punct]
    
    
    dist_jaccard_1 = textdistance.jaccard.normalized_similarity(ent_sw_removed, sim_1_sw_removed)
    dist_jaccard_2 = textdistance.jaccard.normalized_similarity(ent_sw_removed, sim_2_sw_removed)
    dist_jaccard_3 = textdistance.jaccard.normalized_similarity(ent_sw_removed, sim_3_sw_removed)
    
    return dist_jaccard_1, dist_jaccard_2, dist_jaccard_3

In [204]:
def calcular_overlap(fila):

    doc_ent = nlp(fila['Entidad'])
    doc_sim_1 = nlp(fila['similar_1'])
    doc_sim_2 = nlp(fila['similar_2'])
    doc_sim_3 = nlp(fila['similar_3'])
    
    ent_sw_removed = [token.text for token in doc_ent if not token.is_stop and not token.is_punct]
    sim_1_sw_removed = [token.text for token in doc_sim_1 if not token.is_stop and not token.is_punct]
    sim_2_sw_removed = [token.text for token in doc_sim_2 if not token.is_stop and not token.is_punct]
    sim_3_sw_removed = [token.text for token in doc_sim_3 if not token.is_stop and not token.is_punct]
    
    dist_overlap_1 = textdistance.overlap.normalized_similarity(ent_sw_removed, sim_1_sw_removed)
    dist_overlap_2 = textdistance.overlap.normalized_similarity(ent_sw_removed, sim_2_sw_removed)
    dist_overlap_3 = textdistance.overlap.normalized_similarity(ent_sw_removed, sim_3_sw_removed)
    
    return dist_overlap_1, dist_overlap_2, dist_overlap_3

In [205]:
def calcular_distancias(fila):
    doc_ent = nlp(fila['Entidad'])
    doc_sim_1 = nlp(fila['similar_1'])
    doc_sim_2 = nlp(fila['similar_2'])
    doc_sim_3 = nlp(fila['similar_3'])
    
    ent_sw_removed = " ".join([token.text for token in doc_ent if not token.is_stop and not token.is_punct])
    sim_1_sw_removed = " ".join([token.text for token in doc_sim_1 if not token.is_stop and not token.is_punct])
    sim_2_sw_removed = " ".join([token.text for token in doc_sim_2 if not token.is_stop and not token.is_punct])
    sim_3_sw_removed = " ".join([token.text for token in doc_sim_3 if not token.is_stop and not token.is_punct])
    
    dist_jaro_1 = textdistance.jaro_winkler(ent_sw_removed, sim_1_sw_removed)
    dist_jaro_2 = textdistance.jaro_winkler(ent_sw_removed, sim_2_sw_removed)
    dist_jaro_3 = textdistance.jaro_winkler(ent_sw_removed, sim_3_sw_removed)

    dist_leven_1 = textdistance.levenshtein.normalized_similarity(ent_sw_removed, sim_1_sw_removed)
    dist_leven_2 = textdistance.levenshtein.normalized_similarity(ent_sw_removed, sim_2_sw_removed)
    dist_leven_3 = textdistance.levenshtein.normalized_similarity(ent_sw_removed, sim_3_sw_removed)

    dist_jaccard_1 = textdistance.jaccard.normalized_similarity(ent_sw_removed, sim_1_sw_removed)
    dist_jaccard_2 = textdistance.jaccard.normalized_similarity(ent_sw_removed, sim_2_sw_removed)
    dist_jaccard_3 = textdistance.jaccard.normalized_similarity(ent_sw_removed, sim_3_sw_removed)

    dist_overlap_1 = textdistance.overlap.normalized_similarity(ent_sw_removed, sim_1_sw_removed)
    dist_overlap_2 = textdistance.overlap.normalized_similarity(ent_sw_removed, sim_2_sw_removed)
    dist_overlap_3 = textdistance.overlap.normalized_similarity(ent_sw_removed, sim_3_sw_removed)
    
    return dist_jaro_1, dist_jaro_2, dist_jaro_3, dist_leven_1, dist_leven_2, dist_leven_3, dist_jaccard_1, dist_jaccard_2, dist_jaccard_3, dist_overlap_1, dist_overlap_2, dist_overlap_3

In [206]:
def excluir_por_numeros(fila):
    excluir_1 = False
    excluir_2 = False
    excluir_3 = False
    
    regex_romanos = r'\b(v[ii]{0,3}|i[vx]|x[lcvi]{0,4}|i{1,3})\b'
    
    romanos_e = set(re.findall(regex_romanos, fila['Entidad']))
    romanos_1 = set(re.findall(regex_romanos, fila['similar_1']))
    romanos_2 = set(re.findall(regex_romanos, fila['similar_2']))
    romanos_3 = set(re.findall(regex_romanos, fila['similar_3']))
    
    if romanos_e and romanos_1 and romanos_e != romanos_1:
        excluir_1 = True
    if romanos_e and romanos_2 and romanos_e != romanos_2:
        excluir_2 = True
    if romanos_e and romanos_3 and romanos_e != romanos_3:
        excluir_3 = True
    
    regex_numeros = r'\d+'
    numeros_e = set(re.findall(regex_numeros, fila['Entidad']))
    numeros_1 = set(re.findall(regex_numeros, fila['similar_1']))
    numeros_2 = set(re.findall(regex_numeros, fila['similar_2']))
    numeros_3 = set(re.findall(regex_numeros, fila['similar_3']))

    if numeros_e != numeros_1:
        excluir_1 = True
    if numeros_e != numeros_2:
        excluir_2 = True
    if numeros_e != numeros_3:
        excluir_3 = True
    
    return excluir_1, excluir_2, excluir_3

In [207]:
def excluir_por_nombre(fila):
    
    doc_ent = nlp(fila['Entidad'])
    ents_ent = {ent.text: ent.label_ for ent in doc_ent.ents}
    palabras_ent = {ent.text for ent in doc_ent.ents}
    labels_excluir = ["PERSON", "GPE", "ORG"]
    
    exclusion = [False, False, False]
    
    for i in range(1,4):
        if len(fila['Entidad'].split()) == len(fila[f'similar_{i}'].split()):

            doc_sim = nlp(fila[f'similar_{i}'])
            ents_sim = {ent.text: ent.label_ for ent in doc_sim.ents}
            palabras_sim = {ent.text for ent in doc_sim.ents}
            
            diferencias = palabras_ent.symmetric_difference(palabras_sim)

            for palabra_dif in diferencias:
                label_e = ents_ent.get(palabra_dif)
                label_sim = ents_sim.get(palabra_dif)
                if label_e in labels_excluir or label_sim in labels_excluir:
                    exclusion[i-1] = True

    return exclusion

In [208]:
nlp = spacy.load("en_core_web_sm")

In [209]:
database_Neo = "2wiki.prueba.rebel.4"
conn_Neo4j = ConexionNeo4j(database_Neo)

In [210]:
entidades = conn_Neo4j.extraer_all_entidades_neo4j()
len(entidades)

6258

In [211]:
df_final = calcular_rank_B25s(entidades)

In [212]:
df_final['entidad_similar_1_iguales'] = False
df_final['entidad_similar_2_iguales'] = False
df_final['entidad_similar_3_iguales'] = False
df_final.loc[df_final['Entidad'] == df_final['similar_1'], 'entidad_similar_1_iguales'] = True
df_final.loc[df_final['Entidad'] == df_final['similar_2'], 'entidad_similar_2_iguales'] = True
df_final.loc[df_final['Entidad'] == df_final['similar_3'], 'entidad_similar_3_iguales'] = True

In [213]:
# df_final[['score_1_inv', 'score_2_inv', 'score_3_inv']] = df_final.apply(calcular_score_inverso, args = (retriever, entidades), axis=1, result_type='expand')

In [214]:
# df_final[['doc_spacy_ent', 'doc_spacy_1', 'doc_spacy_2', 'doc_spacy_3']] = df_final.apply(get_spacy_docs, axis = 1, result_type='expand')

In [215]:
# df_final[['dist_jaro_1', 'dist_jaro_2', 'dist_jaro_3']] = df_final.apply(calcular_jaro_winkler, axis = 1, result_type='expand')

In [216]:
# df_final[['dist_leven_1', 'dist_leven_2', 'dist_leven_3']] = df_final.apply(calcular_levenshtein, axis = 1, result_type='expand')

In [217]:
# df_final[['dist_jaccard_1', 'dist_jaccard_2', 'dist_jaccard_3']] = df_final.apply(calcular_jaccard, axis = 1, result_type='expand')

In [218]:
# df_final[['dist_overlap_1', 'dist_overlap_2', 'dist_overlap_3']] = df_final.apply(calcular_overlap, axis = 1, result_type='expand')

In [219]:
df_final[['dist_jaro_1', 'dist_jaro_2', 'dist_jaro_3', 'dist_leven_1', 'dist_leven_2', 'dist_leven_3', 'dist_jaccard_1', 'dist_jaccard_2', 'dist_jaccard_3', 'dist_overlap_1', 'dist_overlap_2', 'dist_overlap_3']] = df_final.apply(calcular_distancias, axis = 1, result_type='expand')

In [220]:
df_final[['excluir_num_1', 'excluir_num_2', 'excluir_num_3']] = df_final.apply(excluir_por_numeros, axis = 1, result_type='expand')

In [221]:
df_final[['excluir_nom_1', 'excluir_nom_2', 'excluir_nom_3']] = df_final.apply(excluir_por_nombre, axis = 1, result_type='expand')

In [222]:
# df_final['excluir_por_nombre_propio'] = df_final.apply(es_falso_positivo_por_nombre, axis=1)

# Filtros de fusion

In [223]:
df_final['fusion_1'] = False
df_final['fusion_2'] = False
df_final['fusion_3'] = False

## Filtro 1

In [224]:
filtro_1_1 = (
    (
        (
            (df_final['dist_jaro_1'].between(0.8, 0.999)) & (df_final['dist_leven_1'].between(0.75, 0.999))
            ) |
        (
            (df_final['dist_jaro_1'].between(0.9, 0.999)) & (df_final['dist_leven_1'].between(0.71, 0.999))
            )
        ) &
    (df_final['score_1'] > 4.5) &
    (df_final['excluir_num_1'] == False) &
    (df_final['excluir_nom_1'] == False) &
    (df_final['entidad_similar_1_iguales'] == False)
    )

In [225]:
filtro_1_2 = (
    (
        (
            (df_final['dist_jaro_2'].between(0.8, 0.999)) & (df_final['dist_leven_2'].between(0.75, 0.999))
            ) |
        (
            (df_final['dist_jaro_2'].between(0.9, 0.999)) & (df_final['dist_leven_2'].between(0.71, 0.999))
            )
        ) &
    (df_final['score_2'] > 4.5) &
    (df_final['excluir_num_2'] == False) &
    (df_final['excluir_nom_2'] == False) &
    (df_final['entidad_similar_2_iguales'] == False)
    )

In [226]:
filtro_1_3 = (
    (
        (
            (df_final['dist_jaro_3'].between(0.8, 0.999)) & (df_final['dist_leven_3'].between(0.75, 0.999))
            ) |
        (
            (df_final['dist_jaro_3'].between(0.9, 0.999)) & (df_final['dist_leven_3'].between(0.71, 0.999))
            )
        ) &
    (df_final['score_3'] > 4.5) &
    (df_final['excluir_num_3'] == False) &
    (df_final['excluir_nom_3'] == False) &
    (df_final['entidad_similar_3_iguales'] == False)
    )

In [227]:
df_final.loc[filtro_1_1, 'fusion_1'] = True
df_final.loc[filtro_1_2, 'fusion_2'] = True
df_final.loc[filtro_1_3, 'fusion_3'] = True

## Filtro 2

In [228]:
filtro_2_1 = (
    (df_final['dist_jaro_1'].between(0.9, 0.999)) &
    (df_final['dist_leven_1'].between(0.8, 0.999)) &
    (df_final['score_1'] > 1) &
    (df_final['excluir_num_1'] == False) &
    (df_final['excluir_nom_1'] == False) &
    (df_final['entidad_similar_1_iguales'] == False)
    )

In [229]:
filtro_2_2 = (
    (df_final['dist_jaro_2'].between(0.9, 0.999)) &
    (df_final['dist_leven_2'].between(0.8, 0.999)) &
    (df_final['score_2'] > 2) &
    (df_final['excluir_num_2'] == False) &
    (df_final['excluir_nom_2'] == False) &
    (df_final['entidad_similar_2_iguales'] == False)
    )

In [230]:
filtro_2_3 = (
    (df_final['dist_jaro_3'].between(0.9, 0.999)) &
    (df_final['dist_leven_3'].between(0.8, 0.999)) &
    (df_final['score_3'] > 2) &
    (df_final['excluir_num_3'] == False) &
    (df_final['excluir_nom_3'] == False) &
    (df_final['entidad_similar_3_iguales'] == False)
    )

In [231]:
df_final.loc[filtro_2_1, 'fusion_1'] = True
df_final.loc[filtro_2_2, 'fusion_2'] = True
df_final.loc[filtro_2_3, 'fusion_3'] = True

## Filtro 3

In [232]:
filtro_3_1 = (
    (df_final["dist_overlap_1"] == 1) &
    (df_final["dist_jaro_1"] >= 0.9) &
    (df_final["score_1"] >= 4.5) &
    (df_final['excluir_num_1'] == False) &
    (df_final['excluir_nom_1'] == False) &
    (df_final['entidad_similar_1_iguales'] == False)
    )

In [233]:
filtro_3_2 = (
    (df_final["dist_overlap_2"] == 1) &
    (df_final["dist_jaro_2"] >= 0.9) &
    (df_final["score_2"] >= 4.5) &
    (df_final['excluir_num_2'] == False) &
    (df_final['excluir_nom_2'] == False) &
    (df_final['entidad_similar_2_iguales'] == False)
    )


In [234]:
filtro_3_3 = (
    (df_final["dist_overlap_3"] == 1) &
    (df_final["dist_jaro_3"] >= 0.9) &
    (df_final["score_3"] >= 4.5) &
    (df_final['excluir_num_3'] == False) &
    (df_final['excluir_nom_3'] == False) &
    (df_final['entidad_similar_3_iguales'] == False)
    )

In [235]:
df_final.loc[filtro_3_1, 'fusion_1'] = True
df_final.loc[filtro_3_2, 'fusion_2'] = True
df_final.loc[filtro_3_3, 'fusion_3'] = True

In [236]:
df_final

,Entidad,similar_1,score_1,similar_2,score_2,similar_3,score_3,entidad_similar_1_iguales,entidad_similar_2_iguales,entidad_similar_3_iguales,...,dist_overlap_3,excluir_num_1,excluir_num_2,excluir_num_3,excluir_nom_1,excluir_nom_2,excluir_nom_3,fusion_1,fusion_2,fusion_3
0,night at the movies,night at the movies,5.382150,a night at the movies,5.382150,saturday night at the movies,4.438026,True,False,False,...,1.000000,False,False,False,False,False,False,False,True,False
1,chicago ten,chicago ten,5.939967,chicago,3.391698,ten ready rifles,2.696221,True,False,False,...,0.285714,False,False,False,False,False,False,False,False,False
2,jack shea,jack shea,6.195229,jack ferver,2.711984,jack macgowran,2.711984,True,False,False,...,0.666667,False,False,False,False,True,True,False,False,False
3,harry johnson,harry johnson,5.582535,emory johnson,2.870550,daniel johnson,2.870550,True,False,False,...,0.692308,False,False,False,False,True,True,False,False,False
4,harsin,harsin,4.153368,harsin county,3.269801,chahār,0.000000,True,False,False,...,0.500000,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6253,mathews,j e mathews,3.974786,mathews,3.974786,travis mathews,3.129210,False,True,False,...,1.000000,False,False,False,False,False,False,False,False,False
6254,romanian,romanian,4.424488,roman,0.000000,greek,0.000000,True,False,False,...,0.200000,False,False,False,False,False,False,False,False,False
6255,chahār,chahār,4.424488,roman,0.000000,greek,0.000000,True,False,False,...,0.200000,False,False,False,False,False,False,False,False,False
6256,greek,greek,4.424488,roman,0.000000,chahār,0.000000,True,False,False,...,0.200000,False,False,False,False,False,False,False,False,False


# UNION FINAL

In [237]:
cols_1 = ['Entidad', 'similar_1','score_1','dist_jaro_1', 'dist_leven_1','dist_jaccard_1', 'dist_overlap_1','fusion_1']
cols_2 = ['Entidad', 'similar_2','score_2','dist_jaro_2', 'dist_leven_2','dist_jaccard_2', 'dist_overlap_2','fusion_2']
cols_3 = ['Entidad', 'similar_3','score_3','dist_jaro_3', 'dist_leven_3','dist_jaccard_3', 'dist_overlap_3','fusion_3']

df_final_fusion_1 = df_final.loc[df_final['fusion_1'] == True][cols_1]
df_final_fusion_2 = df_final.loc[df_final['fusion_2'] == True][cols_2]
df_final_fusion_3 = df_final.loc[df_final['fusion_3'] == True][cols_3]

rename_cols_1 = {'similar_1': "similar",'score_1': "score",'dist_jaro_1': "dist_jaro", 'dist_leven_1': "dist_leven",'dist_jaccard_1': "dist_jaccard", 'dist_overlap_1': "dist_overlap",'fusion_1': "fusion"}
rename_cols_2 = {'similar_2': "similar",'score_2': "score",'dist_jaro_2': "dist_jaro", 'dist_leven_2': "dist_leven",'dist_jaccard_2': "dist_jaccard", 'dist_overlap_2': "dist_overlap",'fusion_2': "fusion"}
rename_cols_3 = {'similar_3': "similar",'score_3': "score",'dist_jaro_3': "dist_jaro", 'dist_leven_3': "dist_leven",'dist_jaccard_3': "dist_jaccard", 'dist_overlap_3': "dist_overlap",'fusion_3': "fusion"}

df_final_fusion_1 = df_final_fusion_1.rename(columns = rename_cols_1)
df_final_fusion_2 = df_final_fusion_2.rename(columns = rename_cols_2)
df_final_fusion_3 = df_final_fusion_3.rename(columns = rename_cols_3)

In [238]:
df_final_fusion = pd.concat([df_final_fusion_1, df_final_fusion_2, df_final_fusion_3], ignore_index=True)

In [239]:
df_final_fusion = df_final_fusion.drop_duplicates(["Entidad", "similar"]).sort_values("Entidad")

In [240]:
df_final_fusion

,Entidad,similar,score,dist_jaro,dist_leven,dist_jaccard,dist_overlap,fusion
536,1934 film of the same title,1934 film,4.548065,0.920000,0.600000,0.600000,1.0,True
232,400 metres,400 metres event,4.987405,0.925000,0.625000,0.625000,1.0,True
328,400 metres event,400 metres,6.048401,0.925000,0.625000,0.625000,1.0,True
235,a b fazle karim chowdhury,a m b fazle karim chowdhury,6.707862,0.871884,0.920000,0.920000,1.0,True
33,a b fazle karim chowdhury,a b m fazle karim chowdhury,6.707862,0.909101,0.920000,0.920000,1.0,True
...,...,...,...,...,...,...,...,...
561,"wyoming county, west virginia",wyoming county,4.923106,0.900000,0.500000,0.500000,1.0,True
156,young philadelphians,the young philadelphians,5.621514,1.000000,1.000000,1.000000,1.0,True
195,yukon – koyukuk census area,"yukon – koyukuk census area, alaska",7.898044,0.956250,0.781250,0.781250,1.0,True
470,"yukon – koyukuk census area, alaska",yukon – koyukuk census area,9.076736,0.956250,0.781250,0.781250,1.0,True


# Obtencion de parejas y gruupos de fusion

In [265]:

entidades_1 = df_final_fusion['Entidad'].tolist()
entidades_2 = df_final_fusion['similar'].tolist()
set_entidades = list(set(entidades_1 + entidades_2)) # PAra buscar nodos con grados muy altos

In [262]:
grupos = []

for item1, item2 in zip(entidades_1, entidades_2):
    encontrados = []
    for grupo in grupos:
        if item1 in grupo or item2 in grupo:
            encontrados.append(grupo)

    if not encontrados:
        grupos.append({item1, item2})
    else:
        nuevo_grupo = {item1, item2}
        for g in encontrados:
            nuevo_grupo.update(g)
            grupos.remove(g)
        grupos.append(nuevo_grupo)

resultado = [list(g) for g in grupos]

[['1934 film', '1934 film of the same title'], ['400 metres', '400 metres event'], ['abd al-muttalib shaybah', 'abd al-muttalib shaybah ibn hashim'], ['abstract art', 'abstract artist'], ['academy award for best picture', 'academy award for best director'], ['adventure film', 'adventure drama film'], ['al-hasan ibn ali', 'al-hasan ibn ali ibn abi talib'], ['alexander of cyprus', 'alexander cyprius'], ['alexander von zemlinsky', 'alexander zemlinsky'], ['american director', 'american film director'], ['andrew ii', 'andrew ii of hungary'], ['andrew murray craven', 'andrew murray'], ['anne, duchess of cumberland', 'anne, duchess of cumberland and strathearn'], ['another latin love song/ miss world', 'another latin love song/miss world', 'another latin love song'], ['bob dylan', 'another side of bob dylan'], ['archibald bulloch roosevelt', 'archibald bulloch', 'archibald bulloch roosevelt jr'], ['australian rules football', 'australian rules', 'australian rules footballer'], ['battle of we

In [263]:
grupos

[{'1934 film', '1934 film of the same title'},
 {'400 metres', '400 metres event'},
 {'abd al-muttalib shaybah', 'abd al-muttalib shaybah ibn hashim'},
 {'abstract art', 'abstract artist'},
 {'academy award for best director', 'academy award for best picture'},
 {'adventure drama film', 'adventure film'},
 {'al-hasan ibn ali', 'al-hasan ibn ali ibn abi talib'},
 {'alexander cyprius', 'alexander of cyprus'},
 {'alexander von zemlinsky', 'alexander zemlinsky'},
 {'american director', 'american film director'},
 {'andrew ii', 'andrew ii of hungary'},
 {'andrew murray', 'andrew murray craven'},
 {'anne, duchess of cumberland', 'anne, duchess of cumberland and strathearn'},
 {'another latin love song',
  'another latin love song/ miss world',
  'another latin love song/miss world'},
 {'another side of bob dylan', 'bob dylan'},
 {'archibald bulloch',
  'archibald bulloch roosevelt',
  'archibald bulloch roosevelt jr'},
 {'australian rules',
  'australian rules football',
  'australian rules 

# Obtener nodos con grados muy altos para prevenir fusion

In [266]:
from neo4j import GraphDatabase
driver = GraphDatabase.driver(
    "bolt://localhost:7687",
    auth=("neo4j", "password"),
    database = "2wiki.prueba.rebel.4",
)

In [278]:
get_n_relaciones = """
UNWIND $entidades as entidad
MATCH (n:Entity {name: entidad})
RETURN n.name AS name, COUNT{(n)--()} AS n_relaciones
"""
records, summary, key = driver.execute_query(get_n_relaciones, entidades = set_entidades)

In [ ]:
records

In [281]:
grados = {rec['name']: rec['n_relaciones'] for rec in records}

In [282]:
grados

{'mazraeh-ye shomali rural district': 4,
 'bomba, the jungle boy': 5,
 'do you remember dolly bell': 1,
 'naghan rural district': 4,
 'a b fazle karim chowdhury': 1,
 'eleni zaude gabre': 1,
 'santa maría': 7,
 'cheshmeh-ye ali mohammad': 4,
 'bill evans': 1,
 'indie rock': 5,
 'gus green van sant jr': 7,
 'star trek': 7,
 'dinesh babu': 1,
 'calcasieu parish': 6,
 'me': 6,
 'w augustus barratt': 1,
 'edward iv of england': 18,
 'trinidad perez tecson': 3,
 'electric clouds': 1,
 'philip dunne': 3,
 'mazraeh-ye ali': 11,
 'royal oak': 2,
 'when father was away on business': 9,
 'english name': 4,
 "takeshi's castle wwe raw": 1,
 'theodore roosevelt': 18,
 'knox county, kentucky': 2,
 'archibald bulloch': 10,
 'dinesh baboo': 4,
 'trade registers': 1,
 'felix frankfurter professor of law': 1,
 'the sword stained with royal blood': 8,
 'abm fazle karim chowdhury': 2,
 'the piano tuner has': 1,
 'gertrude of saxony': 4,
 'bill evans trio': 1,
 'motion picture director': 3,
 'odrysian king

In [285]:
max(grupos[0], key=len)

'1934 film of the same title'

In [287]:
fusion_final = {}
for grupo in grupos:
    
    nodo_principal = max(grupo, key = len)
    for ent in grupo:
        if grados[ent] > grados[nodo_principal]:
            nodo_principal = ent
    fusion_final[nodo_principal] = [sec for sec in grupo if sec != nodo_principal]

    
    

In [288]:
fusion_final

{'1934 film': ['1934 film of the same title'],
 '400 metres': ['400 metres event'],
 'abd al-muttalib shaybah ibn hashim': ['abd al-muttalib shaybah'],
 'abstract art': ['abstract artist'],
 'academy award for best picture': ['academy award for best director'],
 'adventure film': ['adventure drama film'],
 'al-hasan ibn ali ibn abi talib': ['al-hasan ibn ali'],
 'alexander of cyprus': ['alexander cyprius'],
 'alexander von zemlinsky': ['alexander zemlinsky'],
 'american director': ['american film director'],
 'andrew ii of hungary': ['andrew ii'],
 'andrew murray': ['andrew murray craven'],
 'anne, duchess of cumberland and strathearn': ['anne, duchess of cumberland'],
 'another latin love song/ miss world': ['another latin love song/miss world',
  'another latin love song'],
 'bob dylan': ['another side of bob dylan'],
 'archibald bulloch': ['archibald bulloch roosevelt',
  'archibald bulloch roosevelt jr'],
 'australian rules football': ['australian rules',
  'australian rules footba

# Fusion en Neo4j

A neo4j no se le puede pasar directamente los diccionarios donde la key es el noso principal. Hay que indicarle con nombre. Por ejempo [{"principal":nodo_princ, "secundarios":lista_secundarios}]

In [292]:
datos_para_neo4j = [
    {
        "nodo_principal": grupo,
        "nodos_a_fusionar": fusion_final[grupo]
    }
    for grupo in fusion_final # La lista generada en el paso anterior
]

In [307]:
datos_para_neo4j

[{'nodo_principal': '1934 film',
  'nodos_a_fusionar': ['1934 film of the same title']},
 {'nodo_principal': '400 metres', 'nodos_a_fusionar': ['400 metres event']},
 {'nodo_principal': 'abd al-muttalib shaybah ibn hashim',
  'nodos_a_fusionar': ['abd al-muttalib shaybah']},
 {'nodo_principal': 'abstract art', 'nodos_a_fusionar': ['abstract artist']},
 {'nodo_principal': 'academy award for best picture',
  'nodos_a_fusionar': ['academy award for best director']},
 {'nodo_principal': 'adventure film',
  'nodos_a_fusionar': ['adventure drama film']},
 {'nodo_principal': 'al-hasan ibn ali ibn abi talib',
  'nodos_a_fusionar': ['al-hasan ibn ali']},
 {'nodo_principal': 'alexander of cyprus',
  'nodos_a_fusionar': ['alexander cyprius']},
 {'nodo_principal': 'alexander von zemlinsky',
  'nodos_a_fusionar': ['alexander zemlinsky']},
 {'nodo_principal': 'american director',
  'nodos_a_fusionar': ['american film director']},
 {'nodo_principal': 'andrew ii of hungary', 'nodos_a_fusionar': ['andr

In [ ]:
query_fusion = """
UNWIND $grupos AS grupo
// 1. Buscamos el nodo principal en la base de datos (reemplaza :Entidad por tu etiqueta real)
MATCH (principal:Entity {name: grupo.nodo_principal})

// 2. Buscamos todos los nodos secundarios que pertenecen a ese mismo grupo
MATCH (secundarios:Entity)
WHERE secundarios.name IN grupo.nodos_a_fusionar 

// 3. Agrupamos los nodos secundarios en una lista
WITH principal, collect(secundarios) AS nodos_secundarios

// 4. Invocamos la función de APOC para fusionarlos todos en el principal
CALL apoc.refactor.mergeNodes([principal] + nodos_secundarios, {
  properties: {
    name: "discard",
    `*`: "combine"
  },
  mergeRels: true
}) YIELD node
RETURN count(node) AS fusiones_realizadas
"""


In [309]:
records, summary, key = driver.execute_query(query_fusion, grupos = datos_para_neo4j)

In [310]:
records

[<Record fusiones_realizadas=265>]

In [75]:
df_final

NameError: name 'df_final_excluidos_nombres' is not defined

# FUNCION GENERAL FUSIONAR NODOS

## Funciones

In [9]:
def calcular_rank_B25s(entidades, n_candidatos):
   entidades_token = bm25s.tokenize(entidades, stopwords="en")
   retriever = bm25s.BM25()
   retriever.index(entidades_token)
   results, scores = retriever.retrieve(entidades_token, k=n_candidatos)
   cols_sim = []
   cols_score = []
   for i in range(1, n_candidatos+1):
      cols_sim.append(f"similar_{i}")
      cols_score.append(f"score_{i}")
      
   df_resultados = pd.DataFrame(data=results, columns=cols_sim)
   df_scores = pd.DataFrame(scores, columns=cols_score)
   df_resultados = df_resultados.map(lambda x: entidades[x])
   df_rank_25 = pd.concat([df_resultados, df_scores], axis=1)
   df_rank_25['Entidad'] = entidades
   return df_rank_25

In [10]:
def encontrar_candidatos_iguales_principal(df, n_candidatos):
    
    for i in range(1, n_candidatos+1):
        df[f"entidad_similar_{i}_iguales"] = False
        df.loc[df["Entidad"] == df[f"similar_{i}"], f"entidad_similar_{i}_iguales"] = True
    return df

In [11]:
def get_columnas(n_candidatos):
    cols_jaro = []
    cols_leven = []
    cols_jaccard = []
    cols_overlap = []
    cols_exc_num = []
    cols_exc_nom = []
    for i in range(1, n_candidatos+1):
        cols_jaro.append(f"dist_jaro_{i}")
        cols_leven.append(f"dist_leven_{i}")
        cols_jaccard.append(f"dist_jaccard_{i}")
        cols_overlap.append(f"dist_overlap_{i}")
        cols_exc_num.append(f"excluir_num_{i}")
        cols_exc_nom.append(f"excluir_nom_{i}")
    cols_distancias = cols_jaro + cols_leven + cols_jaccard + cols_overlap
    
    return cols_distancias, cols_exc_num, cols_exc_nom

In [13]:
def calcular_distancias(fila, n_candidatos):
    doc_ent = nlp(fila['Entidad'])
    ent_sw_removed = " ".join([token.text for token in doc_ent if not token.is_stop and not token.is_punct])
    
    distancias_jaro = []
    distancias_leven = []
    distancias_jaccard = []
    distancias_overlap = []
    for i in range(1, n_candidatos+1):
        doc_sim = nlp(fila[f'similar_{i}'])
        sim_sw_removed = " ".join([token.text for token in doc_sim if not token.is_stop and not token.is_punct])
        dist_jaro = textdistance.jaro_winkler(ent_sw_removed, sim_sw_removed)
        dist_leven = textdistance.levenshtein.normalized_similarity(ent_sw_removed, sim_sw_removed)
        dist_jaccard = textdistance.jaccard.normalized_similarity(ent_sw_removed, sim_sw_removed)
        dist_overlap = textdistance.overlap.normalized_similarity(ent_sw_removed, sim_sw_removed)
        
        distancias_jaro.append(dist_jaro)
        distancias_leven.append(dist_leven)
        distancias_jaccard.append(dist_jaccard)
        distancias_overlap.append(dist_overlap)
    
    return distancias_jaro + distancias_leven + distancias_jaccard + distancias_overlap

In [14]:
def excluir_por_numeros(fila, n_candidatos):
    
    regex_romanos = r"\b(v[ii]{0,3}|i[vx]|x[lcvi]{0,4}|i{1,3})\b"
    romanos_ent = set(re.findall(regex_romanos, fila["Entidad"]))
    
    regex_numeros = r"\d+"
    numeros_ent = set(re.findall(regex_numeros, fila["Entidad"]))
    
    excluir = [False, False, False]
    
    for i in range(1, n_candidatos+1):
        romanos_sim = set(re.findall(regex_romanos, fila[f"similar_{i}"]))
        # if romanos_ent and romanos_sim and romanos_ent != romanos_sim:
        if romanos_ent != romanos_sim:
            excluir[i-1] = True
                
        numeros_sim = set(re.findall(regex_numeros, fila[f"similar_{i}"]))
        if numeros_ent != numeros_sim:
            excluir[i-1] = True
    
    return excluir

In [15]:
def excluir_por_nombre(fila, n_candidatos):
    
    doc_ent = nlp(fila['Entidad'])
    ents_ent = {ent.text: ent.label_ for ent in doc_ent.ents}
    palabras_ent = {ent.text for ent in doc_ent.ents}
    labels_excluir = ["PERSON", "GPE", "ORG"]
    
    exclusion = [False, False, False]
    
    for i in range(1, n_candidatos+1):
        if len(fila['Entidad'].split()) == len(fila[f'similar_{i}'].split()):

            doc_sim = nlp(fila[f'similar_{i}'])
            ents_sim = {ent.text: ent.label_ for ent in doc_sim.ents}
            palabras_sim = {ent.text for ent in doc_sim.ents}
            
            diferencias = palabras_ent.symmetric_difference(palabras_sim)

            for palabra_dif in diferencias:
                label_e = ents_ent.get(palabra_dif)
                label_sim = ents_sim.get(palabra_dif)
                if label_e in labels_excluir or label_sim in labels_excluir:
                    exclusion[i-1] = True

    return exclusion

In [16]:
def filtrar_fusionables(df, n_similares):
    
    for i in range(1, n_similares+1):
        col_jaro = f"dist_jaro_{i}"
        col_leven = f"dist_leven_{i}"
        col_jaccard = f"dist_jaccard_{i}"
        col_overlap = f"dist_overlap_{i}"
        col_score = f"score_{i}"
        
        col_ex_num = f"excluir_num_{i}"
        col_ex_nom = f"excluir_nom_{i}"
        col_sim_igual = f"entidad_similar_{i}_iguales"
        
        col_exclusion = f"exclusion_{i}"
        col_fusion = f"fusion_{i}"
        
        df[col_exclusion] = False
        df.loc[(df[col_ex_num] == True) | (df[col_ex_nom] == True) | (df[col_sim_igual] == True), col_exclusion] = True
        
        filtro_1 = (
            (
                (
                    (df[col_jaro].between(0.8, 0.999)) & (df[col_leven].between(0.75, 0.999))
                    ) |
                (
                    (df[col_jaro].between(0.9, 0.999)) & (df[col_leven].between(0.71, 0.999))
                    )
                ) &
            (df[col_score] > 4.5) &
            (df[col_exclusion] == False) 
            )
        
        filtro_2 = (
            (df[col_jaro].between(0.9, 0.999)) &
            (df[col_leven].between(0.8, 0.999)) &
            (df[col_score] > 2) &
            (df[col_exclusion] == False) 
            )
        
        filtro_3 = (
            (df[col_overlap] == 1) &
            (df[col_jaro] >= 0.9) &
            (df[col_score] >= 4.5) &
            (df[col_exclusion] == False) 
            )
        
        df[col_fusion] = False
        df.loc[filtro_1, col_fusion] = True
        df.loc[filtro_2, col_fusion] = True
        df.loc[filtro_3, col_fusion] = True
        
    return df

In [ ]:
def unir_candidatos(df, n_candidatos):
    df_union = pd.DataFrame()
    for i in range(1, n_candidatos+1):
        cols_union = ['Entidad', f'similar_{i}',f'score_{i}',f'dist_jaro_{i}', f'dist_leven_{i}',f'dist_jaccard_{i}', f'dist_overlap_{i}',f'fusion_{i}']
        df_filt = df.loc[df[f'fusion_{i}'] == True, cols_union]
        rename_cols = {f'similar_{i}': "similar",f'score_{i}': "score",f'dist_jaro_{i}': "dist_jaro", f'dist_leven_{i}': "dist_leven",f'dist_jaccard_{i}': "dist_jaccard", f'dist_overlap_{i}': "dist_overlap",f'fusion_{i}': "fusion"}
        df_filt = df_filt.rename(columns = rename_cols)
        df_union = pd.concat([df_union, df_filt], ignore_index=True)
    df_union = df_union.drop_duplicates(["Entidad", "similar"]).sort_values("Entidad").reset_index(drop=True)
    return df_union

In [18]:
def grupos_fusion(df):
    entidades_1 = df['Entidad'].tolist()
    entidades_2 = df['similar'].tolist()
    grupos = []

    for ent_1, ent_2 in zip(entidades_1, entidades_2):
        encontrados = []
        for grupo in grupos:
            if ent_1 in grupo or ent_2 in grupo:
                encontrados.append(grupo)

        if not encontrados:
            grupos.append({ent_1, ent_2})
        else:
            nuevo_grupo = {ent_1, ent_2}
            for g in encontrados:
                nuevo_grupo.update(g)
                grupos.remove(g)
            grupos.append(nuevo_grupo)

    return [list(g) for g in grupos]

In [19]:
def obtener_grados_nodos(driver, df):
# def obtener_grados_nodos(self, df):
    
    entidades = df['Entidad'].tolist()
    similares = df['similar'].tolist()
    set_entidades = list(set(entidades + similares)) 
    
    query = """
        UNWIND $entidades as entidad
        MATCH (n:Entity {name: entidad})
        RETURN n.name AS name, COUNT{(n)--()} AS n_relaciones
    """
    records, summary, key = driver.execute_query(query, entidades = set_entidades)
    # records, summary, key = self.driver.execute_query(get_n_relaciones, entidades = set_entidades, database_ = self.database)
    grados = {rec['name']: rec['n_relaciones'] for rec in records}
    return grados

In [20]:
def seleccionar_nodo_principal(grupos, grados):
    nodos_fusion = {}
    for grupo in grupos:
        
        nodo_principal = max(grupo, key = len)
        for ent in grupo:
            if grados[ent] > grados[nodo_principal]:
                nodo_principal = ent
        nodos_fusion[nodo_principal] = [sec for sec in grupo if sec != nodo_principal]
        
    nodos_fusion_neo4j = [
        {
            "nodo_principal": grupo,
            "nodos_a_fusionar": nodos_fusion[grupo]
        }
        for grupo in nodos_fusion
    ]
    return nodos_fusion_neo4j

In [21]:
# def fusionar_nodos(self, nodos_fusion):
def fusionar_nodos(driver, nodos_fusion):
    query_fusion = """
        UNWIND $grupos AS grupo
        
        MATCH (principal:Entity {name: grupo.nodo_principal})

        MATCH (secundarios:Entity)
        WHERE secundarios.name IN grupo.nodos_a_fusionar 

        WITH principal, collect(secundarios) AS nodos_secundarios, grupo.nodos_a_fusionar AS names_fusionados

        CALL apoc.refactor.mergeNodes([principal] + nodos_secundarios, {
        properties: {
            name: "discard",
            `\\*`: "combine"
        },
        mergeRels: true
        }) YIELD node

        SET node.fusionados = names_fusionados
        RETURN count(node) AS fusiones_realizadas
    """
    records, summary, key = driver.execute_query(query_fusion, grupos = nodos_fusion)
    # records, summary, key = self.driver.execute_query(query_fusion, grupos = datos_para_neo4j, , database_ = self.database)
    return records[0]["fusiones_realizadas"]

## Ejecucion

In [22]:
nlp = spacy.load("en_core_web_sm")
database_Neo = "2wiki.prueba.rebel.5"
conn_Neo4j = ConexionNeo4j(database_Neo)
driver = GraphDatabase.driver(
    "bolt://localhost:7687",
    auth=("neo4j", "password"),
    database = database_Neo,
)


In [23]:
entidades = conn_Neo4j.extraer_all_entidades_neo4j()
n_candidatos = 3

df_deduplicacion = calcular_rank_B25s(entidades, n_candidatos)

df_deduplicacion = encontrar_candidatos_iguales_principal(df_deduplicacion, n_candidatos)

cols_distancias, cols_exc_num, cols_exc_nom = get_columnas(n_candidatos)

df_deduplicacion[cols_distancias] = df_deduplicacion.apply(calcular_distancias, args=(n_candidatos,), axis = 1, result_type='expand')

df_deduplicacion[cols_exc_num] = df_deduplicacion.apply(excluir_por_numeros, args=(n_candidatos,), axis = 1, result_type='expand')
df_deduplicacion[cols_exc_nom] = df_deduplicacion.apply(excluir_por_nombre, args=(n_candidatos,), axis = 1, result_type='expand')
df_deduplicacion = filtrar_fusionables(df_deduplicacion, n_candidatos)

df_deduplicacion_final = unir_candidatos(df_deduplicacion, n_candidatos)
grupos = grupos_fusion(df_deduplicacion_final)
grados = obtener_grados_nodos(driver, df_deduplicacion_final)
nodos_fusion = seleccionar_nodo_principal(grupos, grados)
nodos_fusionados = fusionar_nodos(driver, nodos_fusion)

In [30]:
df_desambiguacion_final

,Entidad,similar,score,dist_jaro,dist_leven,dist_jaccard,dist_overlap,fusion
1,abd al-muttalib shaybah ibn hashim,abd al-muttalib shaybah,5.715919,0.935294,0.676471,0.676471,1.0,True
3,australian rules football,australian rules footballer,3.132057,0.985185,0.925926,0.925926,1.0,True
5,australian rules footballer,australian rules football,3.132057,0.985185,0.925926,0.925926,1.0,True
0,director general of police in punjab,director general of police,4.682092,0.953333,0.766667,0.766667,1.0,True
4,film director,film and tv director,2.164951,0.916346,0.812500,0.812500,1.0,True
2,national gallery of australia (canberra),national gallery of australia,4.932418,0.948571,0.742857,0.742857,1.0,True


In [26]:
nodos_fusionados

5

## Fusion similares_1

In [25]:
cols_ = ['Entidad', 'similar_1','score_1',
       'dist_jaro_1', 'dist_leven_1',
       'dist_jaccard_1', 'dist_overlap_1',
       'excluir_num_1', 'excluir_nom_1','fusion_1']

sim_1_fus = df_final[df_final['fusion_1'] == True][cols_]
sim_1_fus

NameError: name 'df_final' is not defined

# Revision

## Fusion similares_3

In [241]:
cols_ = ['Entidad', 'similar_3','score_3',
       'dist_jaro_3', 'dist_leven_3',
       'dist_jaccard_3', 'dist_overlap_3',
       'excluir_num_3', 'excluir_nom_3','fusion_3']

sim_3_fus = df_final[df_final['fusion_3'] == True][cols_]
sim_3_fus

,Entidad,similar_3,score_3,dist_jaro_3,dist_leven_3,dist_jaccard_3,dist_overlap_3,excluir_num_3,excluir_nom_3,fusion_3
55,la torre de los suplicios,la torre de suso,5.409472,0.915500,0.600000,0.640000,1.00,False,False,True
64,mazraeh-ye ali,mazraeh-ye ali shafi,4.874711,0.940000,0.700000,0.700000,1.00,False,False,True
87,"st mary's high school, quetta",st mary high school,6.455689,0.946154,0.730769,0.730769,1.00,False,False,True
90,mohammed bin rashid al maktoum,mohammed bin rashid al,6.681687,0.946667,0.733333,0.733333,1.00,False,False,True
94,jack ryan conway,jack conway,5.652337,0.901136,0.687500,0.687500,1.00,False,False,True
...,...,...,...,...,...,...,...,...,...,...
5715,motion picture director,motion picture actor,4.684707,0.934165,0.826087,0.791667,0.95,False,False,True
5774,motion picture actor,motion picture director,4.684707,0.934165,0.826087,0.791667,0.95,False,False,True
5981,children's books,children's book,2.940352,0.985714,0.928571,0.928571,1.00,False,False,True
6054,sierra leonean,sierra leonean army,4.947297,0.947368,0.736842,0.736842,1.00,False,False,True


## Fusion similares_1

In [242]:
cols_ = ['Entidad', 'similar_1','score_1',
       'dist_jaro_1', 'dist_leven_1',
       'dist_jaccard_1', 'dist_overlap_1',
       'excluir_num_1', 'excluir_nom_1','fusion_1']

sim_1_fus = df_final[df_final['fusion_1'] == True][cols_]
sim_1_fus

,Entidad,similar_1,score_1,dist_jaro_1,dist_leven_1,dist_jaccard_1,dist_overlap_1,excluir_num_1,excluir_nom_1,fusion_1
141,sasikumar,j sasikumar,4.153368,0.939394,0.818182,0.818182,1.000000,False,False,True
179,the star of india,star of india,5.522741,1.000000,1.000000,1.000000,1.000000,False,False,True
346,the sword stained with royal blood,sword stained with royal blood,8.201560,1.000000,1.000000,1.000000,1.000000,False,False,True
364,the young philadelphians,young philadelphians,5.621514,1.000000,1.000000,1.000000,1.000000,False,False,True
465,why did i get married,why did i get married?,8.270607,1.000000,1.000000,1.000000,1.000000,False,False,True
486,it goes like this,it goes like it goes,6.027068,0.928571,0.642857,0.642857,1.000000,False,False,True
647,a night at the movies,night at the movies,5.382150,1.000000,1.000000,1.000000,1.000000,False,False,True
659,fazle karim chowdhury,a b m fazle karim chowdhury,6.707862,0.851429,0.840000,0.840000,1.000000,False,False,True
675,cradle of humankind,the cradle of humankind,6.210154,1.000000,1.000000,1.000000,1.000000,False,False,True
732,g marthandan,marthandan,4.153368,0.944444,0.833333,0.833333,1.000000,False,False,True


## Fusion similares_2

In [247]:
cols_ = ['Entidad', 'similar_2','score_2',
       'dist_jaro_2', 'dist_leven_2',
       'dist_jaccard_2', 'dist_overlap_2',
       'excluir_num_2', 'excluir_nom_2','fusion_2']

sim_2_fus = df_final[df_final['fusion_2'] == True][cols_]
sim_2_fus

,Entidad,similar_2,score_2,dist_jaro_2,dist_leven_2,dist_jaccard_2,dist_overlap_2,excluir_num_2,excluir_nom_2,fusion_2
0,night at the movies,a night at the movies,5.382150,1.000000,1.000000,1.000000,1.0,False,False,True
16,salehabad rural district,salehabad district,4.983867,0.927778,0.750000,0.750000,1.0,False,False,True
31,rafe kovich and alison barrington kovich,rafe kovich and alison barrington,11.370578,0.961111,0.805556,0.805556,1.0,False,False,True
36,edward iv of england,"edward iv, king of england",5.534581,0.919251,0.772727,0.772727,1.0,False,False,True
40,mission: impossible,mission: impossible iii,4.791569,0.963636,0.818182,0.818182,1.0,False,False,True
...,...,...,...,...,...,...,...,...,...,...
6176,the piano tuner has,the piano tuner has arrived,6.511972,0.915789,0.578947,0.578947,1.0,False,False,True
6189,eleni zaude gabre,eleni zaude gabre-madhin,6.881523,0.941667,0.708333,0.708333,1.0,False,False,True
6204,stefano dell',stefano dell' arzere,4.849126,0.926316,0.631579,0.631579,1.0,False,False,True
6224,boggs steps out,mr boggs steps out,6.112169,0.867965,0.785714,0.785714,1.0,False,False,True


In [244]:
cols_2 = ['Entidad', 'similar_2','score_2','dist_jaro_2', 'dist_leven_2','dist_jaccard_2', 'dist_overlap_2', 'excluir_num_2', 'excluir_nom_2', 'fusion_2'  ]

In [248]:
df_final_2_NO_fusionar = df_final[(df_final['fusion_2'] == False) & (df_final['dist_jaro_2'] != 1) & (df_final['excluir_num_2'] == False)][cols_2]

In [249]:
df_final_2_NO_fusionar

,Entidad,similar_2,score_2,dist_jaro_2,dist_leven_2,dist_jaccard_2,dist_overlap_2,excluir_num_2,excluir_nom_2,fusion_2
2,jack shea,jack ferver,2.711984,0.842424,0.545455,0.428571,0.666667,False,True,False
3,harry johnson,emory johnson,2.870550,0.712821,0.769231,0.625000,0.769231,False,True,False
4,harsin,harsin county,3.269801,0.892308,0.461538,0.461538,1.000000,False,False,False
5,dance or die,dance,3.646231,0.911111,0.555556,0.555556,1.000000,False,False,False
6,lord of tears,lord salisbury,2.940352,0.849286,0.428571,0.500000,0.800000,False,False,False
...,...,...,...,...,...,...,...,...,...,...
6252,iranian,roman,0.000000,0.676190,0.428571,0.333333,0.600000,False,False,False
6254,romanian,roman,0.000000,0.925000,0.625000,0.625000,1.000000,False,False,False
6255,chahār,roman,0.000000,0.455556,0.000000,0.222222,0.400000,False,False,False
6256,greek,roman,0.000000,0.466667,0.000000,0.111111,0.200000,False,False,False
